In [0]:
# importing required functions

from pyspark.sql import functions as F
from delta.tables import DeltaTable

Using the variables defined in the Utilities notebook

In [0]:
%run "/Workspace/FMCG - Atlikon/consolidated_pipeline/01_Setup/Utilities"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
# Creating Widgets

dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data_Scouce")


In [0]:
# Fetching variables in widgets window and assigining back to another variable

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog,data_source)

In [0]:
# locationg path of the input files

base_path = f's3://learning-atlikon-sportsbar/{data_source}/*.csv'
print(base_path)

In [0]:
# Reading data from specified path to a dataframe

df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn("read_timestamp",F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
# Creating Delta Table with CDF property
df.write.format("delta").mode("overwrite")\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
# Identifying Duplicates
df_duplicates = df_bronze.groupBy("customer_id").count().where("count > 1")
df_duplicates.show()


In [0]:
# Removing Duplicates
print("Row before duplicates dropped:",df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Row after duplicates dropped:",df_silver.count())

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
# Cleaning Leading and trial spaces in Customer name column

df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# typos --> correct name

city_mapping = {
    'Bengalore':'Bengaluru',
    'Bengaluruu':'Bengaluru',

    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',

    'NewDelhee':'New Delhi',
    'NewDelhi':'New Delhi',
    'NewDheli':'New Delhi'
}

allowed = ['Bengaluru','Hyderabad','New Delhi']

In [0]:
df_silver = (df_silver
 .replace(city_mapping,subset=["city"])
 .withColumn(
     "city",
     F.when(F.col("city").isNull(), None)
      .when(F.col("city").isin(allowed),F.col("city"))
     .otherwise(None)
    )
)

# Sanity Check

df_silver.select('city').distinct().show()

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap("customer_name"))
)

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# checking City column with null values

df_silver.filter(F.col('city').isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition','Zenathlete Foods','Primefuel Nutrition','Recovery Lane']

df_silver.filter(
    F.col('customer_name').isin(null_customer_names)
).show(truncate=False)

In [0]:
# Business Confirmation Note: City corrections confirmed by business team

customer_city_fix = {
    # Sprintx Nutrition
    789403:"New Delhi",

    # Zenathlete Foods
    789420:"Bengaluru",

    # Primefuel Nutrition
    789521:'Hyderabad',

    # Recovery Lane
    789603:'Hyderabad'

}

In [0]:
df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ['customer_id','fixed_city']
)
df_fix.show()

In [0]:
df_silver = (df_silver
 .join(df_fix,on="customer_id",how="left")
 .withColumn(
     "city",
     F.coalesce("city","fixed_city") # replace null with fixed city
 )
 .drop('fixed_city')
 )

display(df_silver)

In [0]:
df_silver = (df_silver.
             withColumn(
                 "customer_id",
                 F.col("customer_id").cast('string')
                 )
)

df_silver.printSchema()

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws("-","customer_name",F.coalesce(F.col("city"),F.lit("Unknown")))
    )
    .withColumn("market",F.lit("India"))
    .withColumn("platform",F.lit("Sports Bar"))
    .withColumn("channel",F.lit("Acquisition"))
)

display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


# Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

df_gold = df_silver.select("customer_id","customer_name","city","customer","market","platform","channel")

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
%sql
Merge into fmcg.gold.dim_customers as target
using (select customer_id, customer_name, market, platform, channel from fmcg.gold.sb_dim_customers) as source
on (target.customer_code = source.customer_id)
when matched then update set target.customer = source.customer_name,
target.market = source.market,
target.platform = source.platform,
target.channel = source.channel
when not matched then insert (target.customer_code, target.customer, target.market, target.platform, target.channel) values
(source.customer_id,source.customer_name,source.market,source.platform,source.channel)
